# Probabilistic Graphical Modeling of Biomarker–Inflammation Networks in Pediatric Inflammatory Bowel Disease

Elias Suskind, Dhimant Morakhia, Fatehjot Gogia

Run the cell below each time the notebook is started or restarted to ensure that if you change any code in the library, this notebook will use the latest version of the library code.

In [ ]:
%load_ext autoreload
%autoreload 2
%run src/helper_funcs.py
%run src/bayesian_funcs.py

### Abstract:

IBD is a group of chronic autoimmune inflammatory disorders affecting the gastrointestinal tract<sup>[1]</sup>, and understanding the conditional dependency structure between routine clinical biomarkers and tissue inflammation sites could reveal which markers best capture local disease activity.

We hypothesized that CRP would exhibit the highest betweenness and degree centrality in the inferred biomarker-inflammation network, reflecting its role as a systemic bridge between tissue inflammation sites, with this effect being more pronounced in Crohn's Disease (CD).

We applied three network inference methods, two iterations of Hill Climbing Bayesian Network with BIC and Bdeu scoring respectively and the PC algorithm with kernel-based conditional independence testing, to a pediatric IBD dataset of n=78 patients ranging in age from 2 to 16 years old, using MissForest iterative imputation for missing data and maximum likelihood estimation for left-censored lab values.

Contrary to our hypothesis, ESR emerged as the highest betweenness node across both methods in the PC graphs as well as maintaining a relatively high betweenness and degree in the HillClimb graphs (which represent a potentially less optimal greedy graph). This suggests it may serve as a more sensitive systemic bridge between local tissue inflammation and nutritional markers than CRP in pediatric IBD.

A secondary finding of interest is the consistent emergence of Albumin as a high-betweenness node in the CD subgroup across both methods. While Albumin has historically been used as a nutritional marker, emerging consensus challenges its validity in inflammatory contexts, where its decline reflects acute phase response rather than true nutritional status<sup>[2][3]</sup>. Nevertheless, its consistent network centrality in CD, where transmural inflammation and terminal ileal malabsorption co-occur, suggests it may capture a biologically meaningful intersection of inflammatory and nutritional pathways that warrants further investigation.

### Introduction:

Inflammatory bowel disease (IBD) refers to a group of chronic immune-mediated disorders of the gastrointestinal tract marked by recurrent episodes of intestinal inflammation. The condition arises from dysregulated immune responses to intestinal microbiota in genetically and environmentally susceptible individuals. IBD primarily includes two distinct diseases: ulcerative colitis and Crohn’s disease, which differ in their anatomical distribution and the depth of tissue involvement within the bowel wall.<sup>[1]</sup>

Crohn's disease (CD) can affect any region of the gastrointestinal tract, but has a primary pathology involving the terminal ileum and colon. CD is characterized by patchy, transmural inflammation that penetrates the full thickness of the bowel wall and is often further characterized by phenotype (inflammatory, stricturing, and penetrating). Inflammation in CD often occurs in discontinuous segments of the bowel, producing so-called ‘skip lesions’. In contrast, ulcerative colitis (UC) is restricted to affecting the colon and rectum, and involves continuous inflammation of the mucosal layer leading to ulceration of the colonic mucosa. UC typically begins in the rectum and may remain localized or extend proximally through the sigmoid colon, distal colon, or throughout the entire colon in cases of pancolitis.<sup>[1]</sup> IBD often has an overlapping symptomatic phenotype and is thus often confused with Irritable Bowel Syndrome (IBS); however, they can be differentiated by pathology and etiology. While both have environmental and genetic risk factors, IBD is, as mentioned earlier, an immune-mediated disorder by which inflammation can be seen in biopsy. On the other hand, IBS is a functional disorder without an immune-mediated etiology and is typically diagnosed by symptomatic phenotype.<sup>[1][4]</sup> This distinction is worth mentioning as this makes IBD a more promising candidate for probabilistic graphical analysis as it can be quantified by inflammatory biomarkers.

Pertinent to the dataset used in this study, Pediatric IBD accounts for approximately 25% of all IBD cases and is associated with more aggressive and extensive disease phenotypes. Very Early Onset IBD (VEO-IBD) refers to IBD cases in which the disease onset occurs at or before 6 years old and is associated with immunodeficiency-related etiologies including monogenic immune defects affecting epithelial barrier function, phagocytic activity, and T regulatory cell pathways. While the dataset does extend past the diagnostic cutoff for VEO-IBD, the immunological mechanisms characterized in VEO-IBD may remain relevant to this cohort, as age of clinical diagnosis in pediatric patients often lags significantly behind biological disease onset.<sup>[1][5]</sup> Furthermore, pediatric IBD carries distinct consequences including growth impairment, nutritional deficiencies, and neurodevelopmental burden not observed in adult populations.

In terms of biomarkers, the dataset used contained three notable IBD biomarkers: C Reactive Protein (CRP), Erythrocyte Sedimentation Rate (ESR), and serum Albumin.<sup>[2][3][6]</sup> CRP is an endogenous protein synthesized by the liver in response to IL-6 signalling during acute phase response. Thus, it acts as a specific marker of active inflammation with a short half-life, making it clinically useful for monitoring flare-ups. On the other hand, ESR measures the speed erythrocytes settle out of blood over one hour and increases with higher serum levels of inflammatory proteins (i.e. fibrinogen and immunoglobulins) which cause erythrocyte aggregation. It is thus a non-specific and slow-responding marker which can increase in response to acute inflammation, or many chronic disorders.<sup>[1][6][7]</sup> Albumin is a more convoluted marker; representing the primary protein produced by the liver to maintain blood osmolarity. While Albumin has historically been used as a nutritional marker, emerging consensus challenges its validity in inflammatory contexts, where its decline reflects suppressed liver function caused by acute phase response rather than true nutritional status. This ambiguity is particularly pronounced in IBD, where transmural inflammation and intestinal malabsorption may simultaneously suppress hepatic albumin synthesis and reduce dietary protein uptake, making it difficult to attribute hypoalbuminemia to either cause independently.<sup>[2][3][6]</sup>

This relationship between malnutrition-related and hepatic-supression related hypoalbuminemia is just one of many complex bidirectional relationships shared between IBD and its associated biomarkers. These complex bidirectional relationships makes it difficult to determine from pairwise associations alone whether observed correlations reflect direct biological dependencies or shared downstream effects of disease activity.<sup>[1][2][3]</sup> Unlike pairwise correlation approaches, which cannot distinguish direct biological relationships from spurious correlations driven by shared common causes, network inference methods based on conditional independence testing can recover the true direct dependency structure between biomarkers and inflammation sites, enabling more accurate identification of which associations reflect underlying biological pathways.<sup>[8]</sup> Thus, by using probabilistic graphical models to analyze IBD-related biomarkers across a population, the direct dependency structure between biomarkers and tissue inflammation sites is able to be represented more accurately. This, in turn, enables identification of which clinical markers reflect independent biological signals versus shared downstream effects of disease activity.

In sum, this study aims to use three separate network modelling techniques (BN-BIC, BN-Bdeu, and PC) to better characterize the conditional dependency structure between sites of active inflammation and IBD-related biomarkers, in a complex biological system hallmarked by bidirectional relationships. Out of the three previously mentioned biomarkers of note, we hypothesize that CRP will exhibit the highest betweenness and degree centrality in the inferred biomarker-inflammation network over the slow-responding inflammatory biomarkers related to levels of major serum proteins: ESR and Albumin. This being due to CRP responding rapidly to changes in inflammatory activity and possessing a short half-life, making it a more sensitive indicator of current disease state than slower-responding protein-dependent markers.

### Methodology:

#### Dataset
The dataset used consisted of clinical laboratory measurements and biomarker values collected from 85 pediatric patients—ages 2 to 16—diagnosed with Crohn's disease, ulcerative colitis, or other forms of colitis. The initial dataset contained the following fields: Participant_ID, Procedure_Month, Biopsy_Date, Diagnosis, Duo_Inflammation, Gastric_Inflammation, LeftColon_Inflammation, Rectum_Inflammation, RightColon_Inflammation, TI_Inflammation, PGA, Hematocrit, ESR, CRP, Albumin, Vitamin D, Gender, Race, and Ethnicity. Fields were dropped on two grounds: excessive missingness or analytical irrelevance. Specifically, Rectum_Inflammation (96.5% missing) and RightColon_Inflammation (91.8% missing) were excluded due to insufficient data density. Biopsy_Date, Participant_ID, Procedure_Month, and demographic variables were excluded as non-network nodes, though Diagnosis was retained for cohort stratification. Race and ethnicity were not used to stratify due to the potential for imputed variables to lead to fallacious conclusions regarding those stratifications. The following fields were retained as network inference nodes: Duo_Inflammation, Gastric_Inflammation, LeftColon_Inflammation, TI_Inflammation, PGA, Hematocrit, ESR, CRP, Albumin, and Vitamin D. Additionally, data from patients originally included both stool and blood samples, but only blood samples were analyzed in order to maintain consistency and reduce potentially confounding variables.
Laboratory biomarker values and biopsy inflammation grades were collected at separate clinical timepoints and required integration prior to analysis. Lab draws were matched to biopsy timepoints using nearest-neighbor temporal matching within a ±30-day window per patient; where multiple draws qualified, the temporally closest was selected. This procedure yielded a final analytic dataset of n=78 observations.

#### Data Preprocessing
To preprocess the data, a sequence of steps was undertaken. First, 7 patient records were dropped because they contained no usable observations, yielding a final analytic sample of n=78.  Prior to missing value imputation, left-censored laboratory values, which were initially reported as inequality expressions via string formatting (e.g., CRP reported as "<0.8"), were imputed using maximum likelihood estimation (MLE) under a truncated normal distribution fitted to observed values above the detection limit. This approach is preferable to detection limit substitution or midpoint imputation, both of which introduce systematic bias.
In order to fill remaining missing data cells, a Random Forest regression imputation was used with 100 estimators to refine predictions across 10 iterations (MissForest approach). This method is appropriate for datasets containing mixed ordinal and continuous variables. Missingness was assumed to be Missing At Random (MAR), a reasonable assumption given that missing values in this dataset were primarily attributable to patients entering clinical remission and receiving fewer follow-up measurements, rather than any relationship to outcome severity. 
Afterwards, ordinal inflammation grade columns (Gastric_Inflammation, LeftColon_Inflammation, TI_Inflammation, Duo_Inflammation) and PGA were encoded as integers (0–3) preserving rank order.
Lastly, data preprocessing went along two pathways depending on network inference method. For score-based Bayesian Network methods (BN-BIC and BN-BDeu), continuous biomarker values (Hematocrit, ESR, CRP, Albumin, Vitamin D) were discretized into tertiles (encoded 1, 2, and 3) to satisfy the discrete variable assumption of the pgmpy implementation. For the PC algorithm, biomarker values were retained as continuous variables to enable kernel-based conditional independence testing.

#### Network Structure Learning
Networks were learned separately for the full dataset (n=78) and Crohn's Disease subgroup (n=60). The UC subgroup (n=15) was considered insufficient for reliable independent network inference and is treated as exploratory. 
The three networks illustrate different structural learning approaches. Bayesian Information Criterion (BIC) penalizes model complexity as a function of sample size, favoring sparser graphs that capture only the strongest conditional dependencies while suppressing weak or spurious edges.<sup>[9]</sup> This makes it well-suited for small clinical datasets like the one used in this study. Bayesian Dirichlet equivalent uniform (BDeu) incorporates a uniform prior over network structures, which can recover denser dependency structures than BIC.<sup>[10]</sup> Bootstrap stability analysis revealed substantially lower network stability for BN-BDeu across both cohorts, and its results are therefore reported for completeness but not interpreted as primary findings. Its initial inclusion was in order to provide a counterpart to BIC that it could be compared against in terms of graph structure and stability.
The PC (Peter-Clark) algorithm is a constraint-based method that infers network structure by iteratively testing conditional independence between variable pairs, pruning edges where independence is established.<sup>[8]</sup> Kernel-based Conditional Independence (KCI) testing was used as the independence criterion (α=0.05), which detects nonlinear dependencies between continuous variables without distributional assumptions.<sup>[11]</sup> This outputs a Completed Partially Directed Acyclic Graph (CPDAG), which represents a Markov equivalence class of DAGs. Undirected and bidirected edges were represented as two opposing directed edges for centrality computation, preserving path connectivity while acknowledging directional ambiguity. 
While the score-based methods (BIC and BDeu) suggest a continuous flow of information across clinical markers, the constraint-based PC algorithm finds that many variables are statistically independent of one another, a pattern consistent with limited statistical power at n<100 rather than true biological independence.

#### Network Analysis
Networks were analyzed through degree and betweenness centrality, computed using igraph. Degree centrality measures a node's importance by the number of direct connections it has. In this context, this can represent biomarkers or inflammation sites with broad direct associations across the network. Betweenness centrality measures how often a node acts as a bridge along the shortest path between other nodes, thus identifying variables that mediate information flow between otherwise disconnected modules. In this context, this can represent potential systemic integrators of local tissue inflammation signals. Network graphs were generated using matplotlib and igraph for all three inference methods across both cohorts (six total network plots). Betweenness and degree centrality rankings were visualized as horizontal bar charts, with CRP highlighted for hypothesis evaluation.

#### Bootstrap Stability Analysis
To evaluate the reliability of inferred network structures given the limited sample size, a bootstrap stability analysis was conducted for each inference method. For each of 100 bootstrap iterations, a dataset was resampled with replacement from the analytic cohort. A new network was inferred from each resample using identical parameters to the primary analysis. The number of edges that differ between the bootstrapped and original network (Hamming distance) was computed per iteration. Hamming distance between the original graph and bootstrapped graph across all iterations were visualized as histograms. Edge inclusion frequency across all 100 resamples was visualized as a heatmap. Edges with inclusion frequency ~0.5 were considered stable; edges below this threshold were interpreted with caution. 


### Results:

XYZ

### Discussion:



### References:

[1] - McDowell, C., et al. (2023). Inflammatory bowel disease. In StatPearls. StatPearls Publishing. https://www.ncbi.nlm.nih.gov/books/NBK470312/

[2] - Evans, D. C., et al., & ASPEN Malnutrition Committee. (2021). The use of visceral proteins as nutrition markers: An ASPEN position paper. Nutrition in Clinical Practice, 36(1), 22–28. https://doi.org/10.1002/ncp.10588

[3] - Jensen, G. L., et al. (2019). GLIM criteria for the diagnosis of malnutrition: A consensus report from the global clinical nutrition community. Journal of Parenteral and Enteral Nutrition, 43(1), 32–40. https://doi.org/10.1002/jpen.1440

[4] - Rani, R. A., et al. (2016). Irritable bowel syndrome and inflammatory bowel disease overlap syndrome: Pieces of the puzzle are falling into place. Intestinal Research, 14(4), 297–304. https://doi.org/10.5217/ir.2016.14.4.297

[5] - Zheng, Hengqi B., et al. “The growing need to understand very early onset inflammatory bowel disease.” Frontiers in Immunology, vol. 12, 26 May 2021, https://doi.org/10.3389/fimmu.2021.675186.

[6] - Sakurai, Toshiyuki, and Masayuki Saruta. “Positioning and usefulness of biomarkers in inflammatory bowel disease.” Digestion, vol. 104, no. 1, 18 Nov. 2022, pp. 30–41, https://doi.org/10.1159/000527846.

[7] - Singh, G. (2024). C-reactive protein and erythrocyte sedimentation rate in inflammatory lesions: Erythrocyte sedimentation rate test is essential in a subset of patients. American Journal of Clinical Pathology, 161(3), 311. https://doi.org/10.1093/ajcp/aqad125

[8] - Spirtes, P., et al. (2000). Causation, prediction, and search (2nd ed.). MIT Press.

[9] - Schwarz, Gideon. “Estimating the dimension of a model.” The Annals of Statistics, vol. 6, no. 2, 1 Mar. 1978, https://doi.org/10.1214/aos/1176344136.

[10] - Heckerman, David, et al. “Learning bayesian networks: The combination of knowledge and statistical data.” Machine Learning, vol. 20, no. 3, Sept. 1995, pp. 197–243, https://doi.org/10.1007/bf00994016.

[11] - Zhang, K., et al. (2011). Kernel-based conditional independence test and application in causal discovery. In Proceedings of the 27th Conference on Uncertainty in Artificial Intelligence (UAI). https://doi.org/10.48550/arXiv.1202.3775